# Telco Customer Churn — SQL Analysis

**Project:** Customer Churn Analysis for a Telecom Company  
**Notebook** SQL Analysis & Business Insights

## Objective

In this notebook we load the cleaned dataset into a SQLite database and perform in-depth analytical queries. The goal is to identify key customer segments with high churn risk and quantify the revenue impact of customer churn.

**What this notebook covers:**
1. Create SQLite database and load cleaned data
2. Calculate overall business metrics
3. Analyze churn rate across different customer segments
4. Calculate revenue at risk by key dimensions
5. Identify the most critical customer segments
6. Summarize key business insights

**Business value:**
This analysis helps the company understand which customer groups are most likely to churn and how much revenue is at risk, enabling data-driven retention strategies.

## 1. Setup and Database Connection

In [21]:
from pathlib import Path
import sqlite3
import pandas as pd

# Define project paths
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "telco_churn_cleaned.csv"
SQL_DB_PATH = PROJECT_ROOT / "sql" / "telco_churn.db"

print(f"Project root: {PROJECT_ROOT}")
print(f"Database path: {SQL_DB_PATH}")

Project root: C:\Users\vadum\PycharmProjects\Telco Customer Churn
Database path: C:\Users\vadum\PycharmProjects\Telco Customer Churn\sql\telco_churn.db


### Create SQLite Database and Table

In [22]:
# Connect to SQLite database (creates file if it doesn't exist)
conn = sqlite3.connect(SQL_DB_PATH)
cursor = conn.cursor()

# Drop table if exists (for reproducibility)
cursor.execute("DROP TABLE IF EXISTS customers;")

# Create customers table
create_table_sql = """
CREATE TABLE customers (
    CustomerID TEXT PRIMARY KEY,
    gender TEXT,
    SeniorCitizen TEXT,
    Partner TEXT,
    Dependents TEXT,
    tenure INTEGER,
    PhoneService TEXT,
    MultipleLines TEXT,
    InternetService TEXT,
    OnlineSecurity TEXT,
    OnlineBackup TEXT,
    DeviceProtection TEXT,
    TechSupport TEXT,
    StreamingTV TEXT,
    StreamingMovies TEXT,
    Contract TEXT,
    PaperlessBilling TEXT,
    PaymentMethod TEXT,
    MonthlyCharges REAL,
    TotalCharges REAL,
    Churn TEXT,
    TenureGroup TEXT,
    MonthlyChargesBucket TEXT,
    TotalServices INTEGER,
    HasInternetService TEXT,
    AvgMonthlyCharges REAL,
    IsNewCustomer TEXT
);
"""

cursor.execute(create_table_sql)
conn.commit()

print("Table 'customers' created successfully.")

Table 'customers' created successfully.


### Load Cleaned Data into Database

In [23]:
# Load cleaned CSV
df = pd.read_csv(PROCESSED_DATA_PATH)

# Insert data into SQLite
df.to_sql("customers", conn, if_exists="append", index=False)

# Verify row count
cursor.execute("SELECT COUNT(*) FROM customers;")
print(f"Data loaded successfully. Total rows: {cursor.fetchone()[0]}")

Data loaded successfully. Total rows: 7032


### Helper Function to Run Queries

In [24]:
def run_query(sql: str) -> pd.DataFrame:
    """Execute SQL query and return results as DataFrame"""
    return pd.read_sql(sql, conn)

## 2. Overall Business Metrics

First, we calculate key business metrics to understand the overall scale of churn and its financial impact.

In [25]:
query = """
SELECT 
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS overall_churn_rate,
    ROUND(SUM(MonthlyCharges), 2) AS total_monthly_revenue,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS revenue_at_risk
FROM customers;
"""

run_query(query)

,total_customers,churned_customers,overall_churn_rate,total_monthly_revenue,revenue_at_risk
0,7032,1869,26.58,455661.0,139130.85


## 3. Churn Rate by Customer Segments

We analyze churn rate across key customer dimensions to identify which groups are most at risk.

In [26]:
# Churn Rate by Contract Type
query = """
SELECT 
    Contract,
    COUNT(*) AS customer_count,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;
"""

run_query(query)

,Contract,customer_count,churn_rate
0,Month-to-month,3875,42.71
1,One year,1472,11.28
2,Two year,1685,2.85


In [27]:
# Churn Rate by Tenure Group
query = """
SELECT 
    TenureGroup,
    COUNT(*) AS customer_count,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM customers
GROUP BY TenureGroup
ORDER BY TenureGroup;
"""

run_query(query)

,TenureGroup,customer_count,churn_rate
0,0-1 years,2175,47.68
1,1-2 years,1024,28.71
2,2-3 years,832,21.63
3,3-4 years,762,19.03
4,4-5 years,832,14.42
5,5+ years,1407,6.61


In [28]:
# Churn Rate by Payment Method
query = """
SELECT 
    PaymentMethod,
    COUNT(*) AS customer_count,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate DESC;
"""

run_query(query)

,PaymentMethod,customer_count,churn_rate
0,Electronic check,2365,45.29
1,Mailed check,1604,19.20
2,Bank transfer (automatic),1542,16.73
3,Credit card (automatic),1521,15.25


In [29]:
# Churn Rate by Internet Service
query = """
SELECT 
    InternetService,
    COUNT(*) AS customer_count,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM customers
GROUP BY InternetService
ORDER BY churn_rate DESC;
"""

run_query(query)

,InternetService,customer_count,churn_rate
0,Fiber optic,3096,41.89
1,DSL,2416,19.00
2,No,1520,7.43


## 4. Revenue at Risk Analysis

Here we quantify the financial impact of churn by calculating revenue at risk across different segments.

In [30]:
# Revenue at Risk by Contract Type
query = """
SELECT 
    Contract,
    COUNT(*) AS customer_count,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS revenue_at_risk,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY revenue_at_risk DESC;
"""

run_query(query)

,Contract,customer_count,revenue_at_risk,churn_rate
0,Month-to-month,3875,120847.10,42.71
1,One year,1472,14118.45,11.28
2,Two year,1685,4165.30,2.85


In [31]:
# Revenue at Risk by Tenure Group
query = """
SELECT 
    TenureGroup,
    COUNT(*) AS customer_count,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS revenue_at_risk
FROM customers
GROUP BY TenureGroup
ORDER BY revenue_at_risk DESC;
"""

run_query(query)

,TenureGroup,customer_count,revenue_at_risk
0,0-1 years,2175,68954.25
1,1-2 years,1024,23081.65
2,2-3 years,832,15167.95
3,3-4 years,762,12294.55
4,4-5 years,832,10581.90
5,5+ years,1407,9050.55


## 5. Advanced Segmentation

We analyze churn and revenue impact using combinations of dimensions and additional engineered features.

In [32]:
# Churn Rate and Revenue at Risk by Contract + Payment Method
query = """
SELECT 
    Contract || ' + ' || PaymentMethod AS segment,
    COUNT(*) AS customer_count,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS revenue_at_risk
FROM customers
GROUP BY Contract, PaymentMethod
HAVING customer_count >= 50
ORDER BY revenue_at_risk DESC
LIMIT 15;
"""

run_query(query)

,segment,customer_count,churn_rate,revenue_at_risk
0,Month-to-month + Electronic check,1850,53.73,77315.60
1,Month-to-month + Bank transfer (automatic),589,34.13,15151.95
2,Month-to-month + Mailed check,893,31.58,15044.40
3,Month-to-month + Credit card (automatic),543,32.78,13335.15
4,One year + Electronic check,347,18.44,5826.80
5,One year + Credit card (automatic),398,10.30,3544.65
6,One year + Bank transfer (automatic),391,9.72,3127.15
7,Two year + Bank transfer (automatic),562,3.38,1812.80
8,One year + Mailed check,336,6.85,1619.85
9,Two year + Electronic check,168,7.74,1146.35


In [33]:
# Churn Rate by Number of Subscribed Services
query = """
SELECT 
    TotalServices,
    COUNT(*) AS customer_count,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM customers
GROUP BY TotalServices
ORDER BY TotalServices;
"""

run_query(query)

,TotalServices,customer_count,churn_rate
0,1,1260,10.95
1,2,857,31.04
2,3,846,44.92
3,4,965,36.48
4,5,921,31.38
5,6,906,25.61
6,7,674,22.55
7,8,395,12.41
8,9,208,5.29


In [34]:
# Churn Rate: New Customers vs Others
query = """
SELECT 
    IsNewCustomer,
    COUNT(*) AS customer_count,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM customers
GROUP BY IsNewCustomer;
"""

run_query(query)

,IsNewCustomer,customer_count,churn_rate
0,No,5562,19.51
1,Yes,1470,53.33


## 6. Top Risky Segments

Finally, we identify the customer segments that represent the highest financial risk due to churn.

In [35]:
# Top 10 segments with highest revenue at risk
query = """
SELECT 
    Contract || ' + ' || PaymentMethod AS segment,
    COUNT(*) AS customer_count,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS revenue_at_risk,
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM customers
GROUP BY Contract, PaymentMethod
HAVING customer_count >= 30
ORDER BY revenue_at_risk DESC
LIMIT 10;
"""

run_query(query)

,segment,customer_count,revenue_at_risk,churn_rate
0,Month-to-month + Electronic check,1850,77315.60,53.73
1,Month-to-month + Bank transfer (automatic),589,15151.95,34.13
2,Month-to-month + Mailed check,893,15044.40,31.58
3,Month-to-month + Credit card (automatic),543,13335.15,32.78
4,One year + Electronic check,347,5826.80,18.44
5,One year + Credit card (automatic),398,3544.65,10.30
6,One year + Bank transfer (automatic),391,3127.15,9.72
7,Two year + Bank transfer (automatic),562,1812.80,3.38
8,One year + Mailed check,336,1619.85,6.85
9,Two year + Electronic check,168,1146.35,7.74


In [36]:
# Close the database connection
conn.close()
print("Database connection closed.")

Database connection closed.


## Summary & Key Business Insights

**Key Findings:**

- **Overall churn rate** is approximately **26.5%**, representing significant revenue at risk.
- **Contract type** is the strongest predictor of churn. Month-to-month customers churn at a much higher rate than customers on longer-term contracts.
- **Payment method** and **Internet Service** type also show strong correlation with churn behavior.
- Churn risk is highest among **new customers** (first 6 months) and customers with **fewer subscribed services**.
- The combination of **Month-to-month contract + Electronic check** payment method represents one of the highest-risk segments.

**Business Implications:**

- Focus retention efforts on month-to-month customers, especially during their first year.
- Consider incentives to move customers to longer-term contracts.
- Investigate why customers using electronic check have higher churn rates.
- Develop targeted onboarding programs for new customers.

The insights from this SQL analysis will be used to build the final interactive dashboard in Tableau.